# B5 單字錯誤分析（Colab GPU）

1. 上方選單「執行階段 → 變更執行階段類型」選 **GPU**（T4 就夠；有 Pro 可選 L4／A100 比較快）
2. 「執行階段 → 全部執行」
3. 跑完會下載 `analysis_results.zip`：解壓縮後，到老師後台「錯誤分析 → 上傳在電腦上分析好的檔案」把裡面的 `.json` 全選上傳

單字表：把電腦上 `tools\brain\wordbank\` 裡的 7000 單字 CSV 放進這個資料夾的 `wordbank` 子資料夾（沒有的話照樣能分析，只是不會加課外選項）。

In [ ]:
#@title 1. 連接雲端硬碟、準備程式
import os, sys, glob, shutil
from google.colab import drive
drive.mount('/content/drive')
FOLDER_NAME = 'B5 錯誤分析 Colab'  #@param {type:"string"}
cands = glob.glob('/content/drive/MyDrive/**/' + FOLDER_NAME, recursive=True) or glob.glob('/content/drive/MyDrive/' + FOLDER_NAME)
assert cands, '在雲端硬碟找不到資料夾「' + FOLDER_NAME + '」'
SRC = cands[0]
WORK = '/content/brain'
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(SRC, WORK, ignore=shutil.ignore_patterns('out', '*.ipynb'))
sys.path.insert(0, WORK)
os.chdir(WORK)
bank = [f for f in os.listdir(os.path.join(WORK, 'wordbank')) if f.lower().endswith(('.csv', '.tsv', '.txt'))] if os.path.isdir(os.path.join(WORK, 'wordbank')) else []
print('程式資料夾：', SRC)
print('單字表：', bank or '（沒有：不會加課外選項）')
!nvidia-smi -L || echo '⚠ 沒有 GPU：請到「執行階段 → 變更執行階段類型」選 GPU，再全部執行'

In [ ]:
#@title 2. 安裝套件（第一次約 1 分鐘）
!pip -q install nltk==3.10.3 jieba==0.42.1 lemminflect transformers accelerate
import nltk
for p in ('wordnet', 'cmudict', 'averaged_perceptron_tagger_eng'):
    nltk.download(p, download_dir=os.path.join(WORK, 'nltk_data'), quiet=True)
import torch
print('GPU：', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '沒有（會很慢）')

In [ ]:
#@title 3. 從網站讀單字單元，找出要分析的
import json, urllib.request
import brain
FS = 'https://firestore.googleapis.com/v1/projects/eng-3385e/databases/(default)/documents/public/'
def public(doc_id):
    try:
        raw = json.loads(urllib.request.urlopen(FS + urllib.request.quote(doc_id), timeout=30).read())
        return json.loads(raw['fields']['json']['stringValue'])
    except urllib.error.HTTPError as e:
        if e.code == 404: return None
        raise
ANALYZE_ALL = False  #@param {type:"boolean"}
ONLY_UNITS = ''  #@param {type:"string"}
only = [x.strip() for x in ONLY_UNITS.split(',') if x.strip()]
cfg = public('config')
titles = {u['id']: u.get('title', u['id']) for u in cfg.get('customUnits', [])}
types = {u['id']: u['type'] for u in cfg.get('customUnits', [])}
todo = []
for uid in cfg.get('content', {}):
    if types.get(uid) != 'vocab' or (only and uid not in only):
        continue
    c = public('content_' + uid)
    words = [w for w in ((c or {}).get('data') or {}).get('words', []) if w.get('word')]
    if not words:
        continue
    old = public('analysis_' + uid)
    sig = brain.signature(words)
    state = '已分析' if old and old.get('sig') == sig else ('題目改過' if old else '還沒分析')
    need = ANALYZE_ALL or bool(only) or state != '已分析'
    print(('➜ ' if need else '  ') + titles.get(uid, uid), f'({uid})', len(words), '個字', state)
    if need:
        todo.append((uid, c['data']))
print()
print('要分析', len(todo), '個單元' if todo else '個單元：全部都分析好了（要重做就勾 ANALYZE_ALL 或填 ONLY_UNITS）')

In [ ]:
#@title 4. 分析（和網站的分析引擎同一套步驟）
import random, time
OUT = os.path.join(SRC, 'out')
os.makedirs(OUT, exist_ok=True)
done = []
for i, (uid, data) in enumerate(todo, 1):
    t0 = time.time()
    print(f'[{i}/{len(todo)}] {titles.get(uid, uid)} …', end=' ', flush=True)
    a = brain.analyze(data)
    a['unsafe'] = brain.option_safety(data, a)
    a['extra'] = brain.extra_options(data, a, random.Random(uid), uid=uid)
    a['at'] = int(time.time() * 1000)
    with open(os.path.join(OUT, uid + '.json'), 'w', encoding='utf-8') as f:
        json.dump(a, f, ensure_ascii=False)
    done.append(uid)
    print(f'{len(a["words"])} 個字，{time.time() - t0:.0f} 秒')
print('完成', len(done), '個單元；結果也存在雲端硬碟：', OUT)

In [ ]:
#@title 5. 下載結果（到老師後台「錯誤分析 → 上傳在電腦上分析好的檔案」全選上傳）
import zipfile
from google.colab import files
if done:
    z = '/content/analysis_results.zip'
    with zipfile.ZipFile(z, 'w') as zf:
        for uid in done:
            zf.write(os.path.join(OUT, uid + '.json'), uid + '.json')
    files.download(z)
else:
    print('這次沒有分析任何單元')